# From tweets to press articles

This notebook starts with a file of tweets. It extracts the links shared in the tweets, identifies the links that lead to news websites, and retrieves the text of the articles.

The aim is to prepare the articles for text analysis.

**Steps**

1. Read the tweets.
2. Extract the links from the tweets.
3. Clean the links and identify news websites.
4. Follow shortened links to find the original addresses.
5. Create a table of links.
6. Retrieve the text of the articles.
7. Check the results.
8. Take a first look at the articles.

## 0. Setup

Install the required libraries once with `pip install -r requirements.txt`.

In [41]:
import re
import time
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse

import pandas as pd
import requests
import trafilatura

The main settings are defined here. This makes it easier to change them without looking through the rest of the notebook.

In [43]:
# A URL starts with http:// or https:// and runs until a space or a quote.
url_pattern = re.compile(r"https?://[^\s\"'<>]+")

# Punctuation that often sticks to the end of a link inside a sentence.
trailing_punctuation = ".,;:!?)]}…»"

# Query parameters that only track the reader and do not change the page.
tracking_prefixes = ("utm_", "fbclid", "gclid", "xtor", "at_", "mc_")

# Some link shorteners do not redirect with a normal HTTP redirection: they
# return a small page that sends the browser to the real address, either with
# a <meta http-equiv="refresh"> tag or with a line of JavaScript.
meta_refresh = re.compile(r'<meta[^>]+http-equiv=["\']?refresh["\']?[^>]*?content=["\']?\s*(\d+)\s*;\s*url=([^"\'>\s]+)', re.IGNORECASE)
js_redirect = re.compile(r'location(?:\.href)?(?:\.replace)?\s*[=(]\s*["\']([^"\']+)["\']', re.IGNORECASE)
og_url = re.compile(r'<meta[^>]+property=["\']og:url["\'][^>]*?content=["\']([^"\']+)', re.IGNORECASE)
canonical = re.compile(r'<link[^>]+rel=["\']canonical["\'][^>]*?href=["\']([^"\']+)', re.IGNORECASE)
tags_and_scripts = re.compile(r"<script.*?</script>|<style.*?</style>|<[^>]+>", re.IGNORECASE | re.DOTALL)
max_hops = 5          # maximum number of such pages followed for one link

# The program introduces itself like a browser; many sites refuse other clients.
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/130.0 Safari/537.36",
           "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
           "Accept-Language": "fr,en;q=0.8"}
timeout = 20          # seconds before giving up on a request
pause = 1.0           # seconds between two requests, to stay polite
min_characters = 200  # below this length, the page is not treated as an article

# When a site refuses the request, look for a copy of the page in the
# Internet Archive (Wayback Machine), a public archive of the web.
use_archive = True
wayback_api = "https://archive.org/wayback/available"
archive_timeout = 60  # the archive is slower than a news site
archive_attempts = 2  # try twice before giving up
archive_pause = 30 

# Sentences that announce a paywall: the text retrieved is then only the free part.
paywall_phrases = ("réservé aux abonnés", "réservé à nos abonnés", "réservé à nos lecteurs abonnés",
                   "cet article est réservé", "abonnez-vous pour lire", "subscribers only",
                   "subscribe to continue", "réservée aux abonnés", "de cet article à lire")

# Lines that some sites insert in the middle of the text and that are not part of the article.
boilerplate_lines = {"passer la publicité", "passer la publicité passer la publicité", "publicité", "contribuer", "réutiliser ce contenu",}

tweets_file = "data/sample_tweets.csv"
domains_file = "news_domains.txt"
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

pages = {}            # pages already downloaded while resolving links: address -> HTML

## 1. Read the tweets

In [45]:
tweets = pd.read_csv(tweets_file, dtype={"tweet_id": str})
print(len(tweets), "tweets")
tweets.head()

36 tweets


,tweet_id,text
0,2122650056738099473,Soldats français déployés en Ukraine: Moscou y...
1,2122193545864024243,URGENT - Mission «Volia»: comment l'armée fra...
2,2122419594593722383,"Douze médailles, un engouement populaire: le S..."
3,2116938613921349941,Jeune Afrique vous dévoile les coulisses de no...
4,2122490894008062017,MUST READ paper by @Policy_Exchange on the dan...


## 2. Extract the links from the tweets

I use a regular expression to find links starting with `http://` or `https://`. I also remove punctuation at the end of a link. If the same link appears more than once in a tweet, I keep it only once.

In [47]:
def extract_urls(text):
    """Return the list of URLs found in a tweet, in order of appearance."""
    if not isinstance(text, str):
        return []
    urls = []
    for match in url_pattern.findall(text):
        url = match.rstrip(trailing_punctuation)
        if url not in urls:
            urls.append(url)
    return urls


# Example
extract_urls("A lire : https://t.co/AbC123, et aussi (https://www.lemonde.fr/a/b.html).")

['https://t.co/AbC123', 'https://www.lemonde.fr/a/b.html']

## 3. Clean the links and identify news websites

The same article can be shared with different tracking information in its URL, such as `utm_source` or `fbclid`. I remove these parameters so that I can identify the same article even when it is shared in different ways.

I then compare each website with the list in `news_domains.txt`. This list can be updated when needed.

In [49]:
def clean_url(url):
    """Remove tracking parameters and the fragment, so that the same article
    shared in two different ways is counted once."""
    parts = urlparse(url)
    query = [(key, value) for key, value in parse_qsl(parts.query, keep_blank_values=True)
             if not key.lower().startswith(tracking_prefixes)]
    return urlunparse((parts.scheme, parts.netloc.lower(), parts.path, parts.params,
                       urlencode(query), ""))


def get_domain(url):
    """Return the site of a URL without 'www.' or 'm.' in front."""
    domain = urlparse(url).netloc.lower()
    for prefix in ("www.", "m.", "amp."):
        if domain.startswith(prefix):
            domain = domain[len(prefix):]
    return domain


# Example
clean_url("https://WWW.Lemonde.fr/politique/article.html?utm_source=twitter&page=2#comments")

'https://www.lemonde.fr/politique/article.html?page=2'

In [50]:
def load_news_domains(path):
    """Read the list of news sites, one domain per line; '#' starts a comment."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    return {line.strip().lower() for line in lines if line.strip() and not line.startswith("#")}


def is_news(domain, news_domains):
    """True if the domain, or the site it belongs to, is in the list.
    Example: 'sport.francetvinfo.fr' matches 'francetvinfo.fr'."""
    return any(domain == news or domain.endswith("." + news) for news in news_domains)


news_domains = load_news_domains(domains_file)
print(len(news_domains), "news sites in the list")

42 news sites in the list


## 4. Follow shortened links

Some links, such as `t.co` links, do not show the original article address. They first redirect to another address. Some news websites also use their own shortened links.

The function below follows these redirects to find the final address. It also checks a few types of redirects that may appear inside the page itself.

If a website refuses the request, I keep the address reached and record the status. This is useful because I can still identify the website, even when I cannot access the page.

In [52]:
def find_html_redirect(html, current_url):
    """Return the address a page redirects to, or None if it is a normal page."""
    target = None
    match = meta_refresh.search(html[:5000])
    if match and int(match.group(1)) <= 5:          # immediate refresh, not a page that reloads itself
        target = match.group(2)
    else:
        # The other signs only count on a page with almost no visible text:
        # real articles also contain scripts and metadata, which must not be
        # mistaken for a redirection.
        visible_text = tags_and_scripts.sub(" ", html).strip()
        if len(visible_text) < 200:
            # 1. a refresh tag anywhere in the page; 2. a line of JavaScript;
            # 3. the address the page declares as its own (og:url, canonical):
            #    shorteners such as ebx.sh copy it from the article they point to.
            for pattern, group in ((meta_refresh, 2), (js_redirect, 1), (og_url, 1), (canonical, 1)):
                match = pattern.search(html)
                if match:
                    target = match.group(group)
                    break
    if target is None:
        return None
    target = target.replace("\\/", "/").replace("&amp;", "&")
    return urljoin(current_url, target)


def resolve_url(url, session):
    """Follow a link to its final address and return (final_url, status)."""
    current = url
    for _ in range(max_hops):
        try:
            response = session.get(current, allow_redirects=True, timeout=timeout, headers=headers)
        except requests.RequestException as error:
            return current, type(error).__name__
        current = response.url                      # address after HTTP redirections
        if response.status_code >= 400:
            return current, f"http_{response.status_code}"
        next_url = find_html_redirect(response.text, current)
        if next_url is None or next_url == current:
            pages[clean_url(current)] = response.text
            return current, "ok"
        current = next_url                          # the page was only a redirection
    return current, "too_many_redirects"


# Example: a shortener page of the kind returned by t.co
find_html_redirect('<head><meta http-equiv="refresh" content="0;URL=https://www.rfi.fr/fr/afrique/article"></head>',
                   "https://t.co/AbC123")

'https://www.rfi.fr/fr/afrique/article'

I use these examples to check that the functions work as expected. These checks do not require an internet connection.

In [54]:
assert extract_urls("Fin https://t.co/AbC123. Encore https://t.co/AbC123") == ["https://t.co/AbC123"]
assert extract_urls("pas de lien ici") == []
assert clean_url("https://www.lemonde.fr/a.html?utm_source=tw&id=7#top") == "https://www.lemonde.fr/a.html?id=7"
assert get_domain("https://m.lefigaro.fr/a") == "lefigaro.fr"
assert is_news("sport.francetvinfo.fr", {"francetvinfo.fr"})
assert not is_news("notlemonde.fr", {"lemonde.fr"})
assert find_html_redirect('<script>location.replace("https:\\/\\/go.rfi.fr\\/vq9")</script>', "https://t.co/x") == "https://go.rfi.fr/vq9"
assert find_html_redirect("<html><body><p>Un article normal.</p></body></html>", "https://www.rfi.fr/a") is None
assert find_html_redirect("<article>" + "<p>Texte de l article.</p>" * 30 + "<script>location.href='/x'</script></article>", "https://www.rfi.fr/a") is None
interstitial = '<html><head><script>dataLayer=[{}]</script><title></title><meta property="og:description" content="Résumé de l article" /><meta property="og:url" content="https://information.tv5monde.com/international/article-123" /></head><body><script>go(target)</script></body></html>'
assert find_html_redirect(interstitial, "https://ebx.sh/EUkUtT") == "https://information.tv5monde.com/international/article-123"
print("All checks passed")

All checks passed


## 5. Create the table of links

I create one row for each link found in a tweet. The table includes the original link, the final address, the website domain and an indication of whether the link leads to a news website.

Each original link is resolved only once.

In [56]:
def build_links_table(tweets, news_domains):
    """One row per (tweet, link): raw link, final link, site, press or not."""
    session = requests.Session()
    resolved = {}                       # raw URL -> (final URL, status), to resolve each link once
    rows = []
    for tweet in tweets.itertuples(index=False):
        for raw_url in extract_urls(tweet.text):
            if raw_url not in resolved:
                resolved[raw_url] = resolve_url(raw_url, session)
                time.sleep(pause)
            final_url, status = resolved[raw_url]
            final_url = clean_url(final_url)
            domain = get_domain(final_url)
            rows.append({"tweet_id": tweet.tweet_id, "raw_url": raw_url, "final_url": final_url,
                         "resolve_status": status, "domain": domain,
                         "is_news": is_news(domain, news_domains)})
    return pd.DataFrame(rows, columns=["tweet_id", "raw_url", "final_url", "resolve_status",
                                       "domain", "is_news"])


links = build_links_table(tweets, news_domains)
links.to_csv(output_folder / "links.csv", index=False)
links

,tweet_id,raw_url,final_url,resolve_status,domain,is_news
0,2122650056738099473,https://t.co/GawJ6iGzgP,https://www.rfi.fr/fr/europe/20261117-soldats-...,http_403,rfi.fr,True
1,2122193545864024243,https://t.co/TXK8EghRnO,https://www.rfi.fr/fr/europe/20261116-mission-...,http_403,rfi.fr,True
2,2122419594593722383,https://t.co/OdSoMQr57Q,https://u.afp.com/SFMj,http_403,u.afp.com,False
3,2116938613921349941,https://x.com/jeune_afrique/status/21169386139...,https://x.com/jeune_afrique/status/21169386139...,ok,x.com,False
4,2122490894008062017,https://t.co/65lS02DGRj,https://policyexchange.org.uk/wp-content/uploa...,ok,policyexchange.org.uk,False
5,2122283520806584577,https://t.co/htAzCZtdFN,https://www.bbc.com/news/articles/c8y5x3n4kywpo,ok,bbc.com,True
6,2122185520256065992,https://t.co/bnL9qlGOyr,https://www.leparisien.fr/faits-divers/larmee-...,http_403,leparisien.fr,True
7,2122613220653293716,https://t.co/GlCmqftiDB,https://www.leparisien.fr/politique/expulser-d...,http_403,leparisien.fr,True
8,2122039291253498341,https://t.co/M9SAoKd5Ue,https://www.leparisien.fr/societe/religions/po...,http_403,leparisien.fr,True
9,2122205908855877835,https://t.co/Y0wBJnb2xC,https://africa24tv.com/sommet-francophonie-2026/,ok,africa24tv.com,True


In [57]:
print("Links found:          ", len(links))
print("Links to a news site: ", int(links["is_news"].sum()))
links["domain"].value_counts().head(15)

Links found:           36
Links to a news site:  29


domain
leparisien.fr            6
lepoint.fr               5
rfi.fr                   4
lefigaro.fr              4
lemonde.fr               3
ebx.sh                   2
lexpress.fr              2
bbc.com                  1
africa24tv.com           1
u.afp.com                1
policyexchange.org.uk    1
youtube.com              1
saharareporters.com      1
tvanouvelles.ca          1
x.com                    1
Name: count, dtype: int64

The domains that are not in my list of news websites can also be checked. Some of them may be news websites that I have not added yet.

In [59]:
links.loc[~links["is_news"], "domain"].value_counts().head(15)

domain
ebx.sh                   2
u.afp.com                1
x.com                    1
policyexchange.org.uk    1
youtube.com              1
ifri.org                 1
Name: count, dtype: int64

## 6. Retrieve the text of the articles

I retrieve each distinct article only once. I use `trafilatura` to extract the title, date and text from the webpage, without keeping most of the menus, adverts and other page elements.

If a website refuses access, I check whether a copy is available in the Internet Archive. I do not try to bypass the website's access restrictions.

The `status` column indicates what happened:

- `ok`: the article text was retrieved from the website;
- `ok_archive`: the text was retrieved from an archived copy;
- `no_text`: no article text could be extracted;
- other values, such as `http_404` or `http_403`, indicate an error or that access was refused.

The `paywall` column indicates whether the text contains a message showing that the article is reserved for subscribers. In that case, the extracted text may be incomplete.

The `archive` column records the result of the search in the Internet Archive.

In [61]:
article_columns = ["url", "status", "title", "date", "n_characters", "paywall", "archive", "text"]


def empty_result(url, status):
    """Row for an article that could not be retrieved."""
    return {"url": url, "status": status, "title": "", "date": "", "n_characters": 0,
            "paywall": False, "archive": "", "text": ""}


def extract_article(url, html):
    """Extract the article from the HTML of a page. Returns a dictionary."""
    # trafilatura separates the article from the menus, adverts and footers.
    article = trafilatura.bare_extraction(html, url=url, favor_precision=True, with_metadata=True)
    if article is not None and not isinstance(article, dict):
        article = article.as_dict()         # recent versions return an object
    text = (article or {}).get("text") or ""
    lines = [line for line in text.splitlines() if line.strip().lower() not in boilerplate_lines]
    text = "\n".join(lines).strip()
    if len(text) < min_characters:
        return empty_result(url, "no_text")
    paywall = any(phrase in text.lower() for phrase in paywall_phrases)
    return {"url": url, "status": "ok", "title": article.get("title") or "",
            "date": article.get("date") or "", "n_characters": len(text), "paywall": paywall,
            "archive": "", "text": text}


def download(url, session):
    """Return (html, status). The page comes from PAGES if it was already downloaded."""
    if url in pages:
        return pages[url], "ok"
    try:
        response = session.get(url, timeout=timeout, headers=headers)
    except requests.RequestException as error:
        return None, type(error).__name__
    time.sleep(pause)
    if response.status_code >= 400:
        return None, f"http_{response.status_code}"
    return response.text, "ok"


def download_from_archive(url, session):
    """
    Look for a copy of a page in the Internet Archive.
    Returns (html, note); html is None when no copy could be retrieved and
    the note says why: 'no_copy', 'http_503', 'error_Timeout', ...
    """
    note = "no_copy"
    for _ in range(archive_attempts):
        try:
            answer = session.get(wayback_api, params={"url": url}, timeout=archive_timeout).json()
            snapshot = answer.get("archived_snapshots", {}).get("closest")
            if not snapshot or not snapshot.get("available"):
                return None, "no_copy"
            # Adding 'id_' after the date asks for the page as it was saved,
            # without the banner that the archive adds on top.
            archive_url = snapshot["url"].replace(snapshot["timestamp"], snapshot["timestamp"] + "id_", 1)
            response = session.get(archive_url, timeout=archive_timeout, headers=headers)
            time.sleep(pause)
            if response.status_code < 400:
                return response.text, "used"
            note = f"http_{response.status_code}"
        except (requests.RequestException, ValueError) as error:
            note = "error_" + type(error).__name__
        time.sleep(archive_pause)
    return None, note


def fetch_article(url, resolve_status, session):
    """Retrieve one article: from the site if possible, otherwise from the archive."""
    html, status = (None, resolve_status) if resolve_status != "ok" else download(url, session)
    if html is not None:
        return extract_article(url, html)
    result = empty_result(url, status)
    if use_archive:
        html, note = download_from_archive(url, session)
        result["archive"] = note
        if html is not None:
            from_archive = extract_article(url, html)
            if from_archive["status"] == "ok":
                from_archive.update(status="ok_archive", archive="used")
                return from_archive
            result["archive"] = "no_text"
    return result

In [62]:
def build_articles_table(links, already_done):
    """One row per distinct press article. Articles already extracted in a
    previous run are skipped; failed ones are tried again."""
    news_links = links[links["is_news"]].drop_duplicates("final_url")
    session = requests.Session()
    rows = []
    for link in news_links.itertuples(index=False):
        if link.final_url not in already_done:
            rows.append(fetch_article(link.final_url, link.resolve_status, session))
    return pd.DataFrame(rows, columns=article_columns)


articles_file = output_folder / "articles.csv"
previous = pd.read_csv(articles_file) if articles_file.exists() else None
if previous is not None:
    previous = previous[previous["status"].str.startswith("ok")]     # keep successes, retry the rest
already_done = set(previous["url"]) if previous is not None else set()

new_articles = build_articles_table(links, already_done)
articles = pd.concat([previous, new_articles], ignore_index=True) if previous is not None else new_articles
articles = articles[articles["url"].isin(links.loc[links["is_news"], "final_url"])].reset_index(drop=True)
articles["paywall"] = articles["text"].fillna("").str.lower().map(lambda t: any(p in t for p in paywall_phrases))
articles.to_csv(articles_file, index=False)
articles.drop(columns="text")

,url,status,title,date,n_characters,paywall,archive
0,https://www.bbc.com/news/articles/c8y5x3n4kywpo,ok,South Africa shootings: 17 killed in Pietermar...,2026-11-16,3152,False,NaN
1,https://africa24tv.com/sommet-francophonie-2026/,ok,Sommet Francophonie 2026,2026-11-13,1805,False,NaN
2,https://www.lefigaro.fr/sports/football/presid...,ok,Élection FIFA : «Quelle farce ! Une honte pour...,2026-11-17,4064,False,NaN
3,https://www.lefigaro.fr/elections/presidentiel...,ok,Présidentielle 2027: Marine Le Pen veut suppri...,2026-11-16,2794,False,NaN
4,https://www.lexpress.fr/monde/asie/en-asie-aus...,ok,"En Asie aussi, les incursions d'avions russes ...",2026-11-17,3333,False,NaN
5,https://www.lexpress.fr/monde/europe/cinq-pays...,ok,Cinq pays de l'UE dévoilent leur projet de cen...,2026-11-16,5827,False,NaN
6,https://www.latribune.fr/article/tech/56008811...,ok,Jusqu’à 489 euros pour changer un écran : ce g...,2026-11-17,4208,True,NaN
7,https://www.lemonde.fr/afrique/article/2022/11...,ok_archive,Emmanuel Macron dénonce le « projet de prédati...,2022-11-21,2718,False,used
8,https://www.lefigaro.fr/vox/monde/les-chretien...,ok,«Les chrétiens algériens sont une source de pa...,2024-04-23,5033,False,NaN
9,https://www.lefigaro.fr/international/le-gouve...,ok,Le gouvernement suédois envisage d'expulser le...,2023-11-21,2974,False,NaN


## 7. Check the results

In [64]:
print("Distinct press articles:", len(articles))
articles["status"].value_counts()

Distinct press articles: 29


status
ok_archive    12
ok            10
http_403       7
Name: count, dtype: int64

In [65]:
# Share of articles retrieved, by news site
if len(articles):
    by_site = (articles.assign(domain=articles["url"].map(get_domain),
                               retrieved=articles["status"].str.startswith("ok"))
               .groupby("domain")["retrieved"].agg(articles="size", retrieved="sum"))
    display(by_site.sort_values("articles", ascending=False))
    print("Articles retrieved:", int(articles["status"].str.startswith("ok").sum()), "of", len(articles))
    print("Of which behind a paywall (partial text):", int(articles["paywall"].sum()))

,articles,retrieved
domain,,
leparisien.fr,6,3
lepoint.fr,5,3
lefigaro.fr,4,4
rfi.fr,4,3
lemonde.fr,3,3
lexpress.fr,2,2
africa24tv.com,1,1
bbc.com,1,1
latribune.fr,1,1


Articles retrieved: 22 of 29
Of which behind a paywall (partial text): 2


In [66]:
# First lines of one article
ok = articles[articles["status"].str.startswith("ok")]
if len(ok):
    first = ok.iloc[0]
    print(first["title"], "|", first["date"])
    print(first["text"][:600])

South Africa shootings: 17 killed in Pietermaritzburg and Pretoria | 2026-11-16
Two mass shootings in South Africa leave 17 dead
South African police are investigating the murders of 17 people in two separate shootings on Sunday night.
Eight men were fatally shot in a minibus in KwaZulu-Natal province. Police declared them dead when they arrived at the scene. Hours later, nine people were found shot dead inside a shack in western Pretoria.
Police say the two incidents are unrelated. The motive for the shootings has not yet been established.
South Africa is struggling to contain violent crime, including gang-related killings. The country has one of the world's highest murd


## 8. First look at the articles

Now that I have retrieved the article texts, I can start looking at the content of the corpus.

First, I count the words in each article. I then compare the average number of words across news websites. Finally, I look at the words that appear most often in articles mentioning Ukraine.

This is a first descriptive step before going further with the text analysis.

In [68]:

import re
from collections import Counter

# Count the words in each article
def count_words(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\wÀ-ÿ'-]+\b", text.lower()))

articles["n_words"] = articles["text"].fillna("").apply(count_words)

# Average article length by news website
articles["domain"] = articles["url"].apply(get_domain)

articles_by_site = (
    articles[articles["n_words"] > 0]
    .groupby("domain")["n_words"]
    .agg(number_of_articles="count", average_words="mean")
    .sort_values("average_words", ascending=False)
)

display(articles_by_site.round(1))


# Check how many articles mention different news topics
topics = [
    "Ukraine",
    "Russie",
    "Gaza",
    "Israël", 
    "Palestine",
    "États-Unis",
    "Trump",
    "Chine",
    "Soudan",
    "Niger",
    "Burkina Faso",
    "Mali"
]

topic_counts = []

for topic in topics:
    pattern = r"(?<!\w)" + re.escape(topic) + r"(?!\w)"

    matches = (
        articles["title"].fillna("").str.contains(pattern, case=False, regex=True)
        | articles["text"].fillna("").str.contains(pattern, case=False, regex=True)
    )

    topic_counts.append({
        "topic": topic,
        "number_of_articles": int((matches & (articles["n_words"] > 0)).sum())
    })

topic_counts = pd.DataFrame(topic_counts).sort_values(
    "number_of_articles", ascending=False
)

display(topic_counts)


# Select a topic to explore
topic = "Ukraine"

pattern = r"(?<!\w)" + re.escape(topic) + r"(?!\w)"

topic_articles = articles[
    (
        articles["title"].fillna("").str.contains(pattern, case=False, regex=True)
        | articles["text"].fillna("").str.contains(pattern, case=False, regex=True)
    )
    & (articles["n_words"] > 0)
].copy()

print(f"Articles mentioning {topic}: {len(topic_articles)}")


# Count the most frequent words in these articles
# Common French and English words are removed.
stop_words = {
    "avec", "dans", "pour", "mais", "plus", "tout", "tous", "elle", "elles",
    "nous", "vous", "leur", "leurs", "cette", "ces", "comme", "être", "avoir",
    "fait", "faire", "ainsi", "entre", "après", "avant", "alors", "aussi",
    "dont", "quand", "chez", "sans", "sur", "sous", "les", "des", "une",
    "est", "sont", "été", "aux", "qui", "que", "quoi", "pas", "par",
    "the", "and", "for", "with", "that", "this", "from", "was", "were",
    "are", "has", "have", "had", "not", "but", "its", "their", "they",
    "you", "your", "his", "her", "she", "him", "our", "out", "about",
    "into", "over", "after", "before", "than", "then", "also", "can",
    "will", "would", "could", "should", "been", "being", "who", "what",
    "when", "where", "which", "while", "how", "all"
}

words = []

for text in topic_articles["text"].dropna():
    tokens = re.findall(r"\b[a-zA-ZÀ-ÿ]{3,}\b", text.lower())
    words.extend(
        word for word in tokens
        if word not in stop_words and word != topic.lower()
    )

most_common_words = pd.DataFrame(
    Counter(words).most_common(20),
    columns=["word", "frequency"]
)

display(most_common_words)


,number_of_articles,average_words
domain,,
lepoint.fr,3,1228.0
lexpress.fr,2,711.5
latribune.fr,1,656.0
lefigaro.fr,4,579.5
leparisien.fr,3,519.7
lemonde.fr,3,515.3
bbc.com,1,492.0
rfi.fr,3,455.7
africa24tv.com,1,275.0


,topic,number_of_articles
0,Ukraine,4
1,Russie,4
5,États-Unis,4
6,Trump,2
11,Mali,2
2,Gaza,1
3,Israël,1
4,Palestine,1
7,Chine,1
9,Niger,1


Articles mentioning Ukraine: 4


,word,frequency
0,français,19
1,mission,17
2,son,12
3,selon,12
4,pays,12
5,depuis,11
6,soldats,11
7,volia,11
8,deux,10
9,avions,10


## Choices and limits

- The list of news websites is in `news_domains.txt`, so I can add or remove domains when needed.
- I set a pause between requests to avoid sending too many requests to the same websites.
- Some news websites refuse automated access. In these cases, I check the Internet Archive for an available copy. Recent articles may not have been archived yet.
- When an article is behind a paywall, I may only retrieve the part that is freely available. These articles are identified in the `paywall` column.
- Some links may lead to a page listing several articles rather than to one specific article.
- A link may no longer work even if it was available when the tweet was published.
- If the tweets come from the Twitter/X API, the `expanded_url` field may already contain the original address behind a `t.co` link.
- When I run the notebook again, articles that were successfully retrieved are kept, while articles that failed are tried again.